In [1]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd

data = pd.read_csv("Data/train.csv", index_col="id")
data

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
id,,,,,,,,,,,,
0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2185,361,1014.6,23.2,20.6,19.1,19.9,97.0,88.0,0.1,40.0,22.1,1
2186,362,1012.4,17.2,17.3,16.3,15.3,91.0,88.0,0.0,50.0,35.3,1
2187,363,1013.3,19.0,16.3,14.3,12.6,79.0,79.0,5.0,40.0,32.9,1


In [2]:
y = data["rainfall"]
X = data.drop(columns="rainfall").copy()

In [3]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, y_encoded, test_size=.2, random_state=42)

-----

# GBM

In [5]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# Define the model
gb_clf = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=42
)

## Cross-validation

In [6]:
# Perform 5-fold cross-validation and compute accuracy
cv_scores = cross_val_score(gb_clf, X, y_encoded, cv=5, scoring='accuracy')

# Print mean and standard deviation of accuracy
print(f"Cross-validated accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

Cross-validated accuracy: 0.8525 ± 0.0138


## External validation

In [7]:

gb_clf.fit(X_train, Y_train)

# Make predictions on the test set
y_pred = gb_clf.predict(X_test)

# Evaluate accuracy
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(Y_test, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")

Test set accuracy: 0.8539


----

# LdaBoost

In [ ]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

X = X.to_numpy()

# ---------- utilities ----------
def softmax(F):
    expF = np.exp(F - np.max(F, axis=1, keepdims=True))
    return expF / np.sum(expF, axis=1, keepdims=True)


# ---------- model ----------
class CustomMulticlassGradientBoostingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

        # attributes that will hold the learned state (with trailing “_”)
        self.estimators_ = None
        self.lda_transforms_ = None
        self.initial_logit_ = None
        self.classes_ = None

    # -------------------------- training --------------------------
    def fit(self, X, y):
        # reset state
        self.estimators_ = []
        self.lda_transforms_ = []
        self.initial_logit_ = None
        self.classes_ = None

        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        one_hot_y = np.eye(n_classes)[np.searchsorted(self.classes_, y)]

        prior = np.clip(np.mean(one_hot_y, axis=0), 1e-5, 1 - 1e-5)
        self.initial_logit_ = np.log(prior)

        F = np.tile(self.initial_logit_, (n_samples, 1))

        for m in range(self.n_estimators):
            # 1) LDA
            if m == 0:
                lda = LinearDiscriminantAnalysis()
                X_lda = lda.fit_transform(X, y)
            else:
                p = softmax(F)
                resid = one_hot_y - p
                labels = np.argmax(resid, axis=1)
                lda = LinearDiscriminantAnalysis()
                X_lda = lda.fit_transform(X, labels)

            self.lda_transforms_.append(lda)

            # 2) updated residuals
            p = softmax(F)
            resid = one_hot_y - p

            # 3) one tree per class
            estims_m = []
            for k in range(n_classes):
                reg = DecisionTreeRegressor(max_depth=self.max_depth)
                reg.fit(X_lda, resid[:, k])
                estims_m.append(reg)
                F[:, k] += self.learning_rate * reg.predict(X_lda)

            self.estimators_.append(estims_m)

        return self

    # -------------------------- inference --------------------------
    def _decision_function(self, X):
        n_samples = X.shape[0]
        F = np.tile(self.initial_logit_, (n_samples, 1))
        for lda, estims_m in zip(self.lda_transforms_, self.estimators_):
            X_lda = lda.transform(X)
            for k, reg in enumerate(estims_m):
                F[:, k] += self.learning_rate * reg.predict(X_lda)
        return F

    def predict_proba(self, X):
        return softmax(self._decision_function(X))

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# ---------- cross-validation ----------
def cross_validate_model(X, y, model, cv=10):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    accs = []

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        model_fold = clone(model)          # now clone works
        model_fold.fit(X_tr, y_tr)
        accs.append(accuracy_score(y_te, model_fold.predict(X_te)))

    return np.mean(accs), np.std(accs)



model = CustomMulticlassGradientBoostingClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=7
)

mean_acc, std_acc = cross_validate_model(X, y_encoded, model, cv=10)
print(f"Mean accuracy: {mean_acc:.4f} ± {std_acc:.4f}")


Accuratezza media: 0.8174 ± 0.0232


## External validation

In [ ]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()

In [10]:
model = CustomMulticlassGradientBoostingClassifier(
    n_estimators=150, learning_rate=0.054398530041183925, max_depth=2
)

model.fit(X_train, Y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(Y_test, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")

Test set accuracy: 0.8562


-----

# LDA+GBM  and PCA+GBM

## External validation and cross-validation

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Any, Dict

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------------------
# DuoBoostCV — evaluates LDA+GB and PCA+GB, with CV and external test
# ---------------------------------------------------------------------
@dataclass
class DuoBoostCV:
    """Compares LDA+GradientBoost vs PCA+GradientBoost with cross-validation
    and optionally evaluates accuracy on an external dataset.

    Main methods
    ------------
    fit(X, y, **cv_kwargs) -> pd.DataFrame
        Runs cross-validation on the training data.

    evaluate_external(X_train, y_train, X_test, y_test, metric=accuracy_score)
        Trains the two pipelines on (X_train, y_train) and computes the metric
        (accuracy by default) on (X_test, y_test).
    """

    seed: int = 42
    _models: Dict[str, Any] = field(init=False, repr=False)

    # -----------------------------------------------------------
    # Build models based on the number of features
    # -----------------------------------------------------------
    def _build_models(self, n_features: int) -> Dict[str, Any]:
        half_components = max(1, n_features // 2)
        return {
            "PCA + Gradient Boost": Pipeline(
                [
                    ("pca", PCA(n_components=half_components, random_state=self.seed)),
                    ("gb", GradientBoostingClassifier(random_state=self.seed)),
                ]
            ),
            "LDA + Gradient Boost": Pipeline(
                [
                    ("lda", LDA(n_components=None)),
                    ("gb", GradientBoostingClassifier(random_state=self.seed)),
                ]
            ),
        }

    # -----------------------------------------------------------
    # CV on training data
    # -----------------------------------------------------------
    def fit(self, X, y, **cv_kwargs) -> pd.DataFrame:
        X = np.asarray(X)
        y = np.asarray(y)

        cv_kwargs = cv_kwargs.copy()
        cv_kwargs.setdefault("scoring", "accuracy")

        self._models = self._build_models(X.shape[1])
        scores: Dict[str, Dict[str, float]] = {}

        for name, model in self._models.items():
            cv_res = cross_validate(model, X, y, **cv_kwargs)

            # <<< dynamic metric detection >>>
            metric_key = next(k for k in cv_res if k.startswith("test_"))
            metric_name = metric_key.replace("test_", "")

            scores[name] = {
                f"mean_{metric_name}": cv_res[metric_key].mean(),
                f"std_{metric_name}":  cv_res[metric_key].std(),
            }

        self.results_ = pd.DataFrame(scores).T.sort_values(
            by=f"mean_{metric_name}", ascending=False
        )
        return self.results_

    # -----------------------------------------------------------
    # Evaluation on an external dataset
    # -----------------------------------------------------------
    def evaluate_external(
        self,
        X_train,
        y_train,
        X_test,
        y_test,
        metric=accuracy_score,
    ) -> pd.DataFrame:
        """Train each pipeline on the training set and compute the metric
        on the external test set.

        Parameters
        ----------
        X_train, y_train : array-like
            Training data.
        X_test, y_test : array-like
            Test data.
        metric : callable, default=accuracy_score
            A function that takes (y_true, y_pred) and returns a float.

        Returns
        -------
        pd.DataFrame
            DataFrame with a column containing the test performance.
        """
        X_train = np.asarray(X_train)
        y_train = np.asarray(y_train)
        X_test = np.asarray(X_test)
        y_test = np.asarray(y_test)

        self._models = self._build_models(X_train.shape[1])

        scores = {}
        for name, model in self._models.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            scores[name] = metric(y_test, y_pred)

        metric_name = metric.__name__.replace("_score", "")
        self.external_results_ = (
            pd.Series(scores, name=f"test_{metric_name}")
            .to_frame()
            .sort_values(by=f"test_{metric_name}", ascending=False)
        )
        return self.external_results_

    # convenience
    def __call__(self, X, y, **cv_kwargs) -> pd.DataFrame:
        return self.fit(X, y, **cv_kwargs)

    def __repr__(self) -> str:  # pragma: no cover
        rep = f"{self.__class__.__name__}(seed={self.seed})"
        if hasattr(self, "results_"):
            rep += f"\nCV results:\n{self.results_.round(4)}"
        if hasattr(self, "external_results_"):
            rep += f"\nExternal test results:\n{self.external_results_.round(4)}"
        return rep

# ---------------------------------------------------------------------
# Usage example
# ---------------------------------------------------------------------
trainer = DuoBoostCV(seed=42)
cv_results = trainer(X_train, Y_train, cv=10, scoring="accuracy", n_jobs=-1)
print(cv_results)

test_results = trainer.evaluate_external(X_train, Y_train, X_test, Y_test)
print(test_results)

                      mean_score  std_score
LDA + Gradient Boost    0.859597   0.034006
PCA + Gradient Boost    0.857302   0.020231
                      test_accuracy
LDA + Gradient Boost       0.856164
PCA + Gradient Boost       0.835616
